# Colab - Evaluate PEFT model and log metrics to Comet
Notebook nay chay evaluate-only de tinh metric tren val/test va log vao dung Comet experiment key.

In [ ]:
import os
import sys
import json
import shlex
import subprocess
from pathlib import Path

print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# === Edit if needed ===
REPO_DIR = '/content/Capstone'
REPO_URL = ''  # ex: 'https://github.com/<org>/<repo>.git'

EVALUATION_MODEL_ID = 'quangne/text2diagram-AceMath-1.5B-Instruct-merged-geometry3k8-8-1-1'
DATASET_WORKSPACE = 'quangne'
DATASET_REPO = 'geometry3k8-8-1-1'
GENERATION_METRICS_SPLITS = 'val,test'
GENERATION_METRICS_MAX_SAMPLES = '0'
GENERATION_METRICS_MAX_NEW_TOKENS = '256'
COMET_EXPERIMENT_KEY = 'fe664e8758544dc68d5edcdc2d854f45'
COMET_PROJECT_NAME = 'text2diagram-llm'

In [ ]:
repo_path = Path(REPO_DIR)
if not repo_path.exists():
    if not REPO_URL:
        raise RuntimeError('REPO_DIR not found. Set REPO_URL, or mount/copy repo to /content first.')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
print('Has requirements:', Path('pipeline/services/peft_finetuning/requirements.txt').exists())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'pipeline/services/peft_finetuning/requirements.txt'
], check=True)
print('Dependencies installed.')

In [ ]:
# Set COMET_API_KEY from Colab Secrets if available, else keep existing env value
try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    try:
        secret_key = userdata.get('COMET_API_KEY')
        if secret_key:
            os.environ['COMET_API_KEY'] = secret_key
    except Exception:
        pass

if 'COMET_API_KEY' not in os.environ or not os.environ['COMET_API_KEY']:
    raise RuntimeError('Missing COMET_API_KEY. Add it to Colab Secrets or os.environ first.')

os.environ['COMET_PROJECT_NAME'] = COMET_PROJECT_NAME
os.environ['COMET_EXPERIMENT_KEY'] = COMET_EXPERIMENT_KEY

print('COMET_PROJECT_NAME =', os.environ.get('COMET_PROJECT_NAME'))
print('COMET_EXPERIMENT_KEY set:', bool(os.environ.get('COMET_EXPERIMENT_KEY')))

In [ ]:
cmd = [
    sys.executable, '-m', 'pipeline.services.peft_finetuning.finetune',
    '--evaluate_only', 'true',
    '--evaluation_model_id', EVALUATION_MODEL_ID,
    '--dataset_huggingface_workspace', DATASET_WORKSPACE,
    '--dataset_huggingface_repo_name', DATASET_REPO,
    '--compute_generation_metrics', 'true',
    '--generation_metrics_splits', GENERATION_METRICS_SPLITS,
    '--generation_metrics_max_samples', GENERATION_METRICS_MAX_SAMPLES,
    '--generation_metrics_max_new_tokens', GENERATION_METRICS_MAX_NEW_TOKENS,
    '--comet_experiment_key', COMET_EXPERIMENT_KEY,
]

print('Running command:')
print(' '.join(shlex.quote(x) for x in cmd))

result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'Command failed with exit code {result.returncode}')
print('Evaluate-only run completed successfully.')

In [ ]:
metrics_path = Path('outputs/generation_metrics.json')
if metrics_path.exists():
    print('Metrics file:', metrics_path)
    with metrics_path.open('r', encoding='utf-8') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print('No local metrics file found at', metrics_path)